In [9]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine

GOLD_PATH = "../data/gold/"

MYSQL_HOST = os.getenv("MYSQL_HOST")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", 3306))
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE")

connection_string = f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
engine = create_engine(connection_string)



In [ ]:
print("="*60)
print("LOADING GOLD TABLES TO MYSQL")
print("="*60)

# Define tables to load (matching your actual Gold output files)
tables_to_load = [
    ('dim_athlete', 'dim_athlete.csv'),
    ('dim_sport', 'dim_sport.csv'),
    ('dim_region', 'dim_region.csv'),
    ('dim_club', 'dim_club.csv'),
    ('dim_date', 'dim_date.csv'),
    ('fact_results', 'fact_results.csv'),
    ('fact_participation', 'fact_participation.csv')
]

# Load each table
for table_name, filename in tables_to_load:
    filepath = os.path.join(GOLD_PATH, filename)
    
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        print(f"Loading {table_name}: {len(df)} rows, {len(df.columns)} columns")
        
        # Load to MySQL (replace if exists)
        df.to_sql(table_name, con=engine, if_exists='replace', index=False)
        print(f"{table_name} loaded successfully")
    else:
        print(f"File not found: {filename}")

print("\n" + "="*60)
print("MYSQL LOAD COMPLETE")
print("="*60)

LOADING GOLD TABLES TO MYSQL
Loading dim_athlete: 7607 rows, 8 columns
  ✅ dim_athlete loaded successfully
Loading dim_sport: 23 rows, 3 columns
  ✅ dim_sport loaded successfully
Loading dim_region: 25 rows, 3 columns
  ✅ dim_region loaded successfully
Loading dim_club: 436 rows, 8 columns
  ✅ dim_club loaded successfully
Loading dim_date: 9 rows, 6 columns
  ✅ dim_date loaded successfully
Loading fact_results: 112296 rows, 13 columns
  ✅ fact_results loaded successfully
Loading fact_participation: 27874 rows, 11 columns
  ✅ fact_participation loaded successfully

MYSQL LOAD COMPLETE


In [11]:
# Verify tables were created
print("\nVerifying tables in MySQL:")
tables_check = pd.read_sql("SHOW TABLES;", engine)
for table in tables_check.iloc[:, 0]:
    count = pd.read_sql(f"SELECT COUNT(*) as count FROM {table};", engine)['count'].iloc[0]
    print(f"  {table}: {count:,} rows")


Verifying tables in MySQL:
  dim_athlete: 7,607 rows
  dim_club: 436 rows
  dim_date: 9 rows
  dim_region: 25 rows
  dim_sport: 23 rows
  fact_participation: 27,874 rows
  fact_results: 112,296 rows
